### Imports

In [1]:
# from model1404RepoVersion import *
from model1404AFWExport import *
import os

folder_name = '15_06_EXPORT_smoldt'
try:
    os.mkdir(folder_name)
except FileExistsError:
    pass


In [ ]:
# Data
# data_dict = np.load("plain_data/data.pkl.npy", allow_pickle=True).item()
# data_dict = np.load("start_conf/last_values.pkl.npy", allow_pickle=True).item()
# data_dict = (make_stretch_plain, (10, 0.0))
# data = np.load("start_conf/last_values.pkl.npy", allow_pickle=True)
# data_dict = np.load("tenxten_sheet.pkl", allow_pickle=True)
# # data_dict['q'][:] = np.array([0,1,0])
# data_dict = (make_two_particles_on_string, (0,))



# data_dict = np.load('schausers_data_3/data.pkl', allow_pickle=True)
# data_dict['q'][:] = np.array([0,1,0])

In [ ]:
# General parameters
neighbour_type = 'voronoi'
elong_func_type = 'cos'
q_mean = True

#Stretching
stretch_factor = 0.000
stretch_stop_ext = 101.0
just_move_bool = True

#gamma params
# gamma_params = [(1.0, 'locked'), (1.0, 'locked')]
gamma_params = (1.0, 'locked')
gamma_diff = 0.0
gamma_range = 2.0
gamma_update_speed = 1.0

#alpha params
# alpha_params = [[(0.0, 'locked'), (-0.0, 'locked')],
                # [(0.0, 'locked'), (-0.0, 'locked')]]


pol_axis = 2
num_cells = 3

data_filename = folder_name + '/basecase_'
if num_cells == 2:
    data_filename = data_filename + '2_'
elif num_cells == 3:
    data_filename = data_filename + '3_'
if pol_axis == 0:
    data_filename = data_filename +'par'
elif pol_axis == 2:
    data_filename = data_filename +'perp'

data_dict = np.load(data_filename + '/data.pkl', allow_pickle=True)
data_dict = {'x': data_dict['x'][-2],
             'p': data_dict['p'][-2],
             'q': data_dict['q'][-2],
             'p_mask': data_dict['p_mask'][-2]}

if pol_axis == 0:
    par_angle = 30.0
    perp_angle = 0.0
elif pol_axis == 2:
    par_angle = 0.0
    perp_angle = 30.0

alpha_params = [(par_angle, 'locked'), (perp_angle, 'locked')]

alpha_range = 45.0
wedge_pcp = True
debug_wedge = False
old_rotation = False
individual_rotation = False

# name = f'par_{par_angle}_perp_{perp_angle}_qwegde_{wedge_pcp}_indi_{individual_rotation}'
if pol_axis == 0:
    name = f'par{num_cells}'
elif pol_axis == 2:
    name = f'perp{num_cells}'

# name = 'basecase_3_perp'

# Simulation parameters
sim_dict = {
    # Data and output
    'output_folder'     : folder_name + f'/{name}',                         # Output folder for simulation data
    'data'              : data_dict,                                        # Tuple containing data information. Can either be a tuple of form (data_gen, *data_gen_args) to generate the data or tuple (p_mask, x, p, q)
    'yield_every'       : 1,                                               # How often the simulation yields data. 
    'yield_steps'       : 15_000,                                             # How many data-yields we want. Total number of timesteps is yield_every * yield_steps

    # Metasimulational parameters
    'device'            : 'cpu',           # Device to run the simulation on. 'cuda' or 'cpu'
    'dtype'             : torch.float,      # Data type for tensors. Either float32 or float64
    'random_seed'       : 50,               # Random seed for the simulation
    'k'                 : 18,                # Number of nearest neighbors to consider  

    # Proliferation parameters
    'prolif_rate'       : 0.0, #[0.0000, 0.0000],           # Cell division probabilities
    'prolif_delay'      : 0,                # Timesteps before the cells begin proliferating
    'max_cells'         : 20_000,           # Maximum number of cells in the simulation. When this number is reached the simulation terminates

    # Main model parameters
    'lambdas'           :   [0.,  .8, .2, 0.0],
                            # [0., .8, 0.2, 0.0],
                            # [0., .8, 0.2, 0.0]],                      # Lambda parameters for the simulation
    'etas'              : 0.0, #[0.0000, 0.0000],                       # Noise level for the simulation
    'alpha_params'      : alpha_params,                                 # Alpha parameters for the simulation
    'alpha_range'       : alpha_range,                                  # Range for alpha parameters
    'gamma_params'      : gamma_params,                                 # Gamma parameters for the simulation
    'gamma_range'       : gamma_range,                                  # Range for gamma parameters
    'cell_wall_interaction' : 0.0,                                      # Strength of cell-wall interactions. 0 for no interaction, 1 for same strength as cell-cell interactions
    'dt'                : 0.001,                                          # Time step for the simulation
    'nematic_pcp'       : False,                                        # Whether planar cell polarity is nematic (True) or vectorial (False)
    'update_cells_bools': [True, True],                                 # List of booleans determining whether to update the parameters for each cell type
    'wedge_pcp'         : wedge_pcp,                                    # Whether to wedge the cells based on their PCP. Only relevant if update_cells_bools is True
    'individual_rotation' : individual_rotation,                        # Whether to apply the wedging rotations individually to each neighbor based on their relative position, or to apply a single rotation based on the mean neighbor position. Only relevant if wedge_pcp is True
    'old_rotation'      : old_rotation,                                 # Whether to use the old method of calculating the rotated ABP vectors, which was based on a simple tangent calculation, or the new method, which uses Rodrigues' rotation formula. The new method is more accurate and can handle larger angles, but the old method is faster and can be sufficient for small angles. Only relevant if alpha parameters are non-zero
    
    # Boundary parameters                                                                                                                                                                                                                                     
    'bound_type'        : None, #'planes',
    'bound_extents'     : [24, 6],
    'bound_move_times'  : [3_000, 17_000],
    'bound_continuity'  : 5_000,
    'stretch_bound_axis': 0,                # The axis along which the stretch is applied. 0 for x, 1 for y, 2 for z. Only relevant if stretch_factor is not 0

    # Stretching parameters
    'stretch_factor'    : stretch_factor,
    'stretch_stop_ext'  : stretch_stop_ext,
    'just_move_bool'    : just_move_bool,                    # Whether to just move the cells according to the stretching without applying any of the other forces. Useful for debugging the stretching implementation
    'stretch_time_stop' : 20_000,                    # Time step at which to stop the stretching. Only relevant if stretch_factor is not 0
    
    #neighbor debugging params
    'use_q_mean'        : q_mean,                                   # Whether to use the mean of q_i and q_j for interactions or just q_i
    'elong_func_type'   : elong_func_type,                          # Whether to use cos(2theta) or linear function for elongation. 'cos' or 'linear'

    #Means of aligning gammas
    'use_gamma_mean'    : True,                             # Whether to use the mean of gamma_i and gamma_j for interactions or just gamma_i
    'gamma_diff_penalty': gamma_diff,                       # Penalty for differences in gamma between neighboring cells. Only relevant if use_gamma_mean is True
    'gamma_update_speed': gamma_update_speed,               # Speed at which gammas are updated based on the difference between neighboring gammas. Only relevant if use_gamma_mean is True
    'screen_out_defects': False,                    
    # Whether to screen out defects in the neighbor calculations. Only relevant if neighbour_type is 'voronoi'

    # Miscellaneous
    'notes'             : name,  # Notes about the simulation. Will be printed when the simulation starts if verbose and will also be saved in .json output
    'verbose'           : True,                                                                     # Verbosity level for the simulation. Set off when debugging
    }

run_simulation(sim_dict=sim_dict)                                                               # Let's run the simulation

Using input data from dictionary
Starting simulation with notes:
par3
Found max safe batch size: 1024
Using batch size: 1024 for true neighbor search.
Using new rotation method
Found max safe batch size: 1024
Using batch size: 1024 for true neighbor search.
Using new rotation method
Simulation done, saved 15000 datapoints0)   (3 cells)
Took 107.15054678916931 seconds
